# Agent 测试 Notebook

这个 notebook 用来逐个测试 `agent/agents` 里的每个 Agent，并在最后跑完整的 Orchestrator 链路。建议从上到下执行。

In [40]:
import importlib
import json
import sys
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if ROOT.name == "agent":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import agent.config as config_module
import agent.agents.prompt as prompt_module
import agent.tools.factor_knowledge as factor_knowledge_module
import agent.tools.data_loader as data_loader_module
import agent.tools.llm_client as llm_client_module
import agent.tools as tools_module
import agent.agents.context_agent as context_agent_module
import agent.agents.forecast_agent as forecast_agent_module
import agent.agents.generation_agent as generation_agent_module
import agent.agents.intent_parser as intent_parser_module
import agent.agents.search_agent as search_agent_module
import agent.agents as agents_module
import agent.orchestrator as orchestrator_module

importlib.reload(config_module)
importlib.reload(prompt_module)
importlib.reload(factor_knowledge_module)
importlib.reload(data_loader_module)
importlib.reload(llm_client_module)
importlib.reload(tools_module)
importlib.reload(context_agent_module)
importlib.reload(forecast_agent_module)
importlib.reload(generation_agent_module)
importlib.reload(intent_parser_module)
importlib.reload(search_agent_module)
importlib.reload(agents_module)
importlib.reload(orchestrator_module)

from agent.models import AgentRequest, UserType
from agent.agents.context_agent import ContextAgent
from agent.agents.forecast_agent import ForecastAgent
from agent.agents.generation_agent import GenerationAgent
from agent.agents.intent_parser import IntentParser
from agent.agents.intent_parser import (
    DataGranularity,
    DataRequirements,
    ParsedIntent,
    TimeRange,
    TimeRangeType,
    TripPlan,
    TripType,
)
from agent.agents.search_agent import SearchAgent
from agent.orchestrator import Orchestrator
from agent.personas import PersonaType

TEST_QUERY = "我想在 2026-07-25 去 Salzburg，什么时候出发最好？"
TEST_DATE = "2026-07-25"
TEST_ROAD = "A8"
TEST_DESTINATION = "Salzburg"
TEST_HOURS = [7, 8, 9, 16, 17, 18]


def to_jsonable(value: Any) -> Any:
    if is_dataclass(value):
        return to_jsonable(asdict(value))
    if isinstance(value, dict):
        return {key: to_jsonable(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_jsonable(item) for item in value]
    if hasattr(value, "value"):
        return value.value
    return value


def show(title: str, value: Any) -> None:
    print(f"\n{'=' * 10} {title} {'=' * 10}")
    if isinstance(value, str):
        print(value)
    else:
        print(json.dumps(to_jsonable(value), ensure_ascii=False, indent=2))


def show_response(name: str, response) -> None:
    show(name, {
        "success": response.success,
        "message": response.message,
        "data": response.data,
    })

print("Agent test environment ready")

Agent test environment ready


## 1. IntentParser 测试

使用关键词降级模式测试意图解析，不依赖 LLM API key。

In [25]:
intent_parser = IntentParser(use_llm=False)
parsed_intent = intent_parser.parse(TEST_QUERY, UserType.TRAVELER)

show("IntentParser", {
    "persona_type": parsed_intent.persona_type,
    "user_type": parsed_intent.user_type,
    "core_question": parsed_intent.core_question,
    "destination": parsed_intent.destination,
    "road": parsed_intent.road,
    "intent": parsed_intent.intent,
    "time_range": parsed_intent.time_range,
    "data_requirements": parsed_intent.data_requirements,
    "trip_plan": parsed_intent.trip_plan,
})


========== IntentParser ==========
{
  "persona_type": "tourist",
  "user_type": "traveler",
  "core_question": "Tell me what I should do.",
  "destination": "salzburg",
  "road": "A8",
  "intent": "plan",
  "time_range": {
    "type": "custom",
    "start_date": "2026-07-25",
    "end_date": "2026-07-25",
    "duration_days": 1,
    "description": "2026-07-25"
  },
  "data_requirements": {
    "time_range": {
      "type": "custom",
      "start_date": "2026-07-25",
      "end_date": "2026-07-25",
      "duration_days": 1,
      "description": "2026-07-25"
    },
    "granularity": "hourly",
    "hours": [
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19
    ]
  },
  "trip_plan": {
    "trip_type": "round_trip",
    "stay_days": 1
  }
}


## 2. ForecastAgent 测试

分别测试小时级预测和日级预测。小时级用于单日出发建议，日级用于日历/长时间范围。

In [26]:
forecast_agent = ForecastAgent()

hourly_request = AgentRequest(
    query="forecast hourly test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    hours=TEST_HOURS,
    granularity="hourly",
)
hourly_forecast_result = await forecast_agent.process(hourly_request)
show_response("ForecastAgent hourly", hourly_forecast_result)


========== ForecastAgent hourly ==========
{
  "success": true,
  "message": "",
  "data": {
    "mode": "hourly",
    "forecast": {
      "date": "2026-07-25",
      "road": "A8",
      "site_id": "A8_default",
      "predictions": [
        {
          "hour": 7,
          "kfz_h_p10": 2088.0190519585394,
          "kfz_h_p50": 2450.665253467299,
          "kfz_h_p90": 2818.6614530746465,
          "sv_h": 287.6409731211398,
          "v_kfz": 127.5353384621389,
          "congestion_score": 23.4,
          "congestion_level": "light"
        },
        {
          "hour": 7,
          "kfz_h_p10": 1594.9503025184742,
          "kfz_h_p50": 2098.9142870473383,
          "kfz_h_p90": 2439.232773056983,
          "sv_h": 65.10422164135021,
          "v_kfz": 103.36669997920768,
          "congestion_score": 25.7,
          "congestion_level": "light"
        },
        {
          "hour": 7,
          "kfz_h_p10": 1598.9207656457709,
          "kfz_h_p50": 2077.848647105513,
         

In [27]:
daily_request = AgentRequest(
    query="forecast daily test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    start_date="2026-07-25",
    end_date="2026-07-31",
    granularity="daily",
    include_factors=True,
)
daily_forecast_result = await forecast_agent.process(daily_request)
show_response("ForecastAgent daily", daily_forecast_result)


========== ForecastAgent daily ==========
{
  "success": true,
  "message": "",
  "data": {
    "mode": "daily",
    "daily_forecasts": [
      {
        "date": "2026-07-25",
        "road": "A8",
        "direction": "Mch",
        "site_id": "A8_Mch_MQB25_Mch_H",
        "site_name": "MQB25_Mch_H",
        "kfz_h_p10": 58081.0,
        "kfz_h_p50": 70603.0,
        "kfz_h_p90": 83239.0,
        "sv_h_pred": 4287.0,
        "v_kfz_pred": 120.8,
        "interval_width": 25158.0,
        "relative_interval_width": 0.428096,
        "congestion_score": 23.3,
        "congestion_level": "light",
        "reasons": [
          {
            "name": "Historical Traffic Baseline",
            "value": 67.5
          },
          {
            "name": "Weather and Temperature",
            "value": 12.6
          },
          {
            "name": "Date and Time Pattern",
            "value": 9.1
          },
          {
            "name": "Special Events",
            "value": 4.8
      

## 3. ContextAgent 测试

测试离线上下文数据，包括天气、假期、活动、施工、气温/路温和历史小时交通。

In [28]:
context_agent = ContextAgent()
context_request = AgentRequest(
    query="context test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    start_date="2026-07-25",
    end_date="2026-07-27",
    hours=[7, 8, 9],
)
context_result = await context_agent.process(context_request)
show("ContextAgent status", {
    "success": context_result.success,
    "message": context_result.message,
})

if context_result.success:
    context_payload = context_result.data.get("context", {})
    context_counts = {
        key: len(value) if isinstance(value, list) else None
        for key, value in context_payload.items()
        if key != "historical_same_period"
    }
    historical_payload = context_payload.get("historical_same_period", {})
    historical_counts = {
        key: len(value) if isinstance(value, list) else None
        for key, value in historical_payload.items()
    }
    history_factor_types = [
        factor.type for factor in context_result.data.get("factors", [])
        if factor.source == "context_history"
    ]
    show("Context summary", context_result.data.get("summary"))
    show("Context row counts", context_counts)
    show("Historical same-period row counts", historical_counts)
    show("Historical same-period factor types", history_factor_types)


========== ContextAgent status ==========
{
  "success": true,
  "message": ""
}

========== Context summary ==========
{
  "weather_days": 3,
  "holiday_days": 3,
  "construction_days": 3,
  "event_days": 3,
  "temperature_hours": 9,
  "traffic_records": 162,
  "avg_speed_kmh": 107.7,
  "max_hourly_volume": 4992.0,
  "min_air_temp_c": 14.8,
  "max_air_temp_c": 18.3,
  "min_road_temp_c": 18.2,
  "max_road_temp_c": 26.5,
  "historical_same_period": {
    "years": [
      "2023",
      "2024",
      "2025"
    ],
    "weather_days": 9,
    "holiday_days": 9,
    "construction_days": 0,
    "event_days": 9,
    "temperature_hours": 27,
    "traffic_records": 162,
    "avg_speed_kmh": 107.7,
    "max_hourly_volume": 4992.0
  }
}

========== Context row counts ==========
{
  "weather": 3,
  "holiday": 3,
  "events": 3,
  "construction": 3,
  "temperature_road": 9,
  "hourly_traffic": 162
}

========== Historical same-period row counts ==========
{
  "weather": 9,
  "holiday": 9,
  "events"

## 4. SearchAgent 测试

测试 Tavily 搜索 Agent。它会按天气、施工、活动、事故四类生成搜索任务；如果没有 Tavily key，会返回 search plan 和错误提示。

In [29]:
search_agent = SearchAgent()
search_request = AgentRequest(
    query="search real-time factors test",
    date=TEST_DATE,
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
)
search_result = await search_agent.process(search_request)
show_response("SearchAgent", search_result)

if search_result.success:
    show("Search plan", search_result.data.get("search_plan"))
    show("Search errors", search_result.data.get("errors"))


========== SearchAgent ==========
{
  "success": true,
  "message": "",
  "data": {
    "factors": [
      {
        "type": "weather",
        "name": "天气预报搜索",
        "description": "On 2026-07-25, expect heavy snow, freezing rain, and black ice in Munich, Rosenheim, and Salzburg. Severe weather warnings may be in effect. Drive cautiously due to potential road disruptions.",
        "impact": "high",
        "source": "search"
      },
      {
        "type": "construction",
        "name": "A8 施工/封路搜索",
        "description": "The A8 Autobahn between Munich and Salzburg will have roadworks closures in July 2026. The project aims to upgrade the highway with a new alignment. Construction is expected to cause traffic disruptions.",
        "impact": "high",
        "source": "search"
      },
      {
        "type": "event",
        "name": "沿线活动搜索",
        "description": "On July 25, 2026, the Auer Dult festival in Munich will take place. Major events include the FREE & EASY Festiv

## 5. GenerationAgent 测试

使用前面 Agent 的输出生成最终建议。这里复用 `IntentParser` 的解析结果、`ForecastAgent` 的预测和 Context/Search factors。

In [30]:
generation_agent = GenerationAgent()

generation_forecast = None
if hourly_forecast_result.success:
    generation_forecast = hourly_forecast_result.data.get("forecast")

context_factors = context_result.data.get("factors", []) if context_result.success else []
search_factors = search_result.data.get("factors", []) if search_result.success else []

generation_result = await generation_agent.process(
    request=hourly_request,
    parsed_intent=parsed_intent,
    forecast=generation_forecast,
    context_factors=context_factors,
    search_factors=search_factors,
)
show_response("GenerationAgent", generation_result)

if generation_result.success:
    show("Generated advice", generation_result.data.get("advice"))


========== GenerationAgent ==========
{
  "success": true,
  "message": "",
  "data": {
    "persona": "tourist",
    "core_question": "Tell me what I should do.",
    "time_range_type": "short",
    "advice": "### 🧳 驾驶建议\n\n**去Salzburg？**\n\n#### 🚗 去程\n\n由于2026-07-25 A8 施工，拥堵风险较高。\n\n✅ **建议**: 07:30 前出发，可以避开大部分车流\n\n⏱️ 预计行程: 1小时44分钟\n\n#### 🔙 返程\n\n**返程时间建议**:\n- 周日返程：建议 12:00 前出发\n- 避开 15:00-19:00 返城高峰",
    "data": {
      "congestion_level": "high",
      "recommended_time": "07:30",
      "trip_type": "round_trip",
      "generation_prompt": {
        "system": "你是 GenerationAgent，负责把预测、上下文、实时搜索和模型归因转成用户真正能执行的出行建议。\n\n总原则：\n1. 不只回答“堵不堵”，而是回答“这个身份的人现在应该怎么做”。\n2. 先给结论，再给关键原因，再给备选方案或注意事项。\n3. 必须结合用户身份调整解释深度、术语、风险表达和行动建议。\n4. 对多日问题给日历/窗口；对单日问题给小时级建议；对管理者给原因和措施。\n5. 使用 FACTOR_CONTRIBUTIONS.md 的归因知识解释 daily reasons。\n6. Weather and Temperature 对未来日期通常是气候态/季节性，不等同实时天气预报。\n7. Construction Impact 在模型训练中学习有限，施工判断要结合 ContextAgent 和 SearchAgent。\n8. 输出要详细、可执行、不要只给一句泛泛建议。\n\n统一输出结构：\n- 标题：点明身

## 5b. GenerationAgent 日级归因知识测试

测试 `GenerationAgent` 是否读取 `doc/FACTOR_CONTRIBUTIONS.md` 的 7 类归因知识，并把日级预测里的 `reasons` 转成用户可读解释。

In [31]:
daily_time_range = TimeRange(
    type=TimeRangeType.CUSTOM,
    start_date="2026-07-25",
    end_date="2026-07-31",
    duration_days=7,
    description="2026-07-25 至 2026-07-31",
)
daily_parsed_intent = ParsedIntent(
    user_type=UserType.TRAVELER,
    persona_type=PersonaType.TOURIST,
    destination=TEST_DESTINATION,
    road=TEST_ROAD,
    intent="plan",
    core_question="Tell me which day is best.",
    time_range=daily_time_range,
    trip_plan=TripPlan(TripType.ROUND_TRIP, stay_days=2),
    data_requirements=DataRequirements(
        time_range=daily_time_range,
        granularity=DataGranularity.DAILY,
        hours=list(range(6, 22)),
    ),
)
daily_generation_request = AgentRequest(
    query="generation daily factor knowledge test",
    date="2026-07-25",
    road=TEST_ROAD,
    destination=TEST_DESTINATION,
    start_date="2026-07-25",
    end_date="2026-07-31",
    granularity="daily",
    include_factors=True,
)
daily_generation_agent = GenerationAgent()
daily_generation_forecast_result = await forecast_agent.process(daily_generation_request)
daily_generation_result = await daily_generation_agent.process(
    request=daily_generation_request,
    parsed_intent=daily_parsed_intent,
    forecast=daily_generation_forecast_result.data.get("daily_forecasts"),
    context_factors=[],
    search_factors=[],
)

advice = daily_generation_result.data.get("advice", "") if daily_generation_result.success else daily_generation_result.message
show("GenerationAgent factor knowledge", {
    "success": daily_generation_result.success,
    "factor_doc": daily_generation_agent.factor_knowledge.get("doc_path"),
    "doc_loaded": daily_generation_agent.factor_knowledge.get("doc_loaded"),
    "has_factor_section": "模型归因解读" in advice,
    "best_day_factor_explanations": daily_generation_result.data.get("data", {}).get("best_day", {}).get("factor_explanations"),
})
show("Generated factor advice excerpt", advice[:1400])


========== GenerationAgent factor knowledge ==========
{
  "success": true,
  "factor_doc": "/Users/linhan.li/Desktop/Autobahn-hackathon/doc/FACTOR_CONTRIBUTIONS.md",
  "doc_loaded": true,
  "has_factor_section": true,
  "best_day_factor_explanations": [
    {
      "name": "Weather and Temperature",
      "value": 9.3,
      "label": "天气和温度",
      "short_name": "WE",
      "baseline": false,
      "usual_range": "通常 3-8%，冬季可更高",
      "message": "天气和温度 贡献约 9.3%。天气/温度是有意义的修正因子，但未来日期通常表示季节性气候特征，不等于实时天气预报。",
      "explanation": "包含气温、路温、降水、降雪、低能见度和结冰风险；未来日期多是气候态平均。",
      "caution": "不要把高 WE 直接理解为确定会有恶劣天气；它更多表示该月份/小时的典型天气影响。"
    },
    {
      "name": "Date and Time Pattern",
      "value": 5.9,
      "label": "日期和时间规律",
      "short_name": "CA",
      "baseline": false,
      "usual_range": "通常 4-15%",
      "message": "日期和时间规律 贡献约 5.9%。日期位置本身在推动车流，比如夏季周末、周五或特定月份节奏。",
      "explanation": "纯日历和时间信号，包括小时、星期、月份、季节和周末模式。",
      "caution": ""
    },
    {
      "name": "Special Events

## 6. 完整 Agent 链路测试

这里直接调用 `Orchestrator`，完整执行 IntentParser → ForecastAgent / ContextAgent / SearchAgent → GenerationAgent。

In [41]:
orchestrator = Orchestrator()
FULL_CHAIN_QUERY = "我想在 2026-07-25 到 2026-07-31 去 Salzburg，哪天出发最好？往返住2天"
chain_result = await orchestrator.process(FULL_CHAIN_QUERY, UserType.TRAVELER)
full_chain_advice = chain_result.get("advice") or ""

show("Full chain query", FULL_CHAIN_QUERY)
show("Full chain success", chain_result.get("success"))
show("Full chain persona", chain_result.get("persona"))
show("Full chain time range", chain_result.get("time_range"))
show("Full chain checks", {
    "has_calendar": "出行日历" in full_chain_advice,
    "has_hourly_context_advice": "每小时建议" in full_chain_advice,
    "has_factor_section": "模型归因解读" in full_chain_advice,
    "return_after_outbound": "2026-07-26" not in full_chain_advice.split("#### 🔙 返程建议")[-1],
})
show("Full chain advice", full_chain_advice)
show("Full chain raw keys", list(chain_result.get("raw", {}).keys()))

[Orchestrator] Persona: traveler
[Orchestrator] Core Question: Which day should we travel?
[Orchestrator] Time Range: custom (2026-07-25 到 2026-07-31)
[Orchestrator]   - Start: 2026-07-25
[Orchestrator]   - End: 2026-07-31
[Orchestrator]   - Duration: 7 days
[Orchestrator] Granularity: daily
[Orchestrator] Trip Type: round_trip
[Orchestrator] Stay Days: 2
[Orchestrator] Running agents: forecast, context, search

========== Full chain query ==========
我想在 2026-07-25 到 2026-07-31 去 Salzburg，哪天出发最好？往返住2天

========== Full chain success ==========
true

========== Full chain persona ==========
{
  "type": "traveler",
  "core_question": "Which day should we travel?"
}

========== Full chain time range ==========
{
  "type": "custom",
  "start_date": "2026-07-25",
  "end_date": "2026-07-31",
  "duration_days": 7,
  "description": "2026-07-25 到 2026-07-31",
  "granularity": "daily"
}

========== Full chain checks ==========
{
  "has_calendar": true,
  "has_hourly_context_advice": true,
  "has_

In [33]:
from agent import config
from agent.tools import LLMClient

show("LLM GPT config", {
    "provider": config.LLM_PROVIDER,
    "model": config.OPENAI_MODEL,
    "openai_key_configured": bool(config.OPENAI_API_KEY),
    "base_url_configured": bool(config.OPENAI_BASE_URL),
})

llm_client = LLMClient()
llm_text = await llm_client.generate(
    "Reply with exactly: GPT_OK",
    system="You are a minimal API connectivity test.",
    temperature=0,
    max_tokens=20,
)
llm_json = await llm_client.generate_json(
    'Return a JSON object with keys status and provider. status must be "ok" and provider must be "gpt".',
    system="You only return JSON.",
    temperature=0,
)

show("LLM GPT interface test", {
    "success": True,
    "model": llm_client.model,
    "text_response": llm_text.strip(),
    "json_response": llm_json,
    "text_ok": "GPT_OK" in llm_text,
    "json_ok": llm_json.get("status") == "ok" and llm_json.get("provider") == "gpt",
})


========== LLM GPT config ==========
{
  "provider": "openai",
  "model": "gpt-4.1-mini",
  "openai_key_configured": true,
  "base_url_configured": false
}

========== LLM GPT interface test ==========
{
  "success": true,
  "model": "gpt-4.1-mini",
  "text_response": "GPT_OK",
  "json_response": {
    "status": "ok",
    "provider": "gpt"
  },
  "text_ok": true,
  "json_ok": true
}
